In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
geoclaw_cube=np.load("/Users/seansantellanes/Downloads/geoclaw_toy_database.npz")
geoclaw_data=geoclaw_cube["eta"][:-1]
hysea_cube=np.load("/Users/seansantellanes/Downloads/combined_eta_thousand.npz")
hysea_data=hysea_cube["eta"][:]
hysea_data=hysea_data.reshape(980,480,356)
hysea_data=np.transpose(hysea_data,(0,2,1))
print(geoclaw_data.shape)
print(hysea_data.shape)

(1000, 356, 480)
(980, 356, 480)


In [4]:
from scipy.linalg import svd
from scipy.spatial.distance import cdist

def energy_distance(X, Y):
    dxy = cdist(X, Y).mean()
    dxx = cdist(X, X).mean()
    dyy = cdist(Y, Y).mean()

    return 2*dxy - dxx - dyy
def compute_2d_eofs(X, n_space=20, n_time=20):
    """
    X : (N, stations, time)

    Returns
    -------
    U : spatial EOFs
    V : temporal EOFs
    mean_field
    """
    mean_field = X.mean(axis=0)
    Xc = X - mean_field

    # Spatial covariance
    Cspace = np.zeros((X.shape[1], X.shape[1]))

    # Temporal covariance
    Ctime = np.zeros((X.shape[2], X.shape[2]))

    for sample in Xc:
        Cspace += sample @ sample.T
        Ctime += sample.T @ sample

    Cspace /= X.shape[0]
    Ctime /= X.shape[0]

    U, _, _ = svd(Cspace)
    V, _, _ = svd(Ctime)

    return (
        U[:, :n_space],
        V[:, :n_time],
        mean_field,
    )
def project_2d(X, U, V, mean_field):

    Xc = X - mean_field

    coeffs = []

    for sample in Xc:

        # coefficient matrix
        C = U.T @ sample @ V

        coeffs.append(C.ravel())

    return np.asarray(coeffs)
def eof2d_energy_test(
        A,
        B,
        n_space=20,
        n_time=20,
        n_permutations=1000,
        random_state=None):

    rng = np.random.default_rng(random_state)

    X = np.concatenate((A, B), axis=0)

    U, V, mean_field = compute_2d_eofs(
        X,
        n_space=n_space,
        n_time=n_time,
    )

    ZA = project_2d(A, U, V, mean_field)
    ZB = project_2d(B, U, V, mean_field)

    observed = energy_distance(ZA, ZB)

    Z = np.vstack((ZA, ZB))
    nA = len(ZA)
    nB = len(ZB)

    perm = np.empty(n_permutations)

    idx = np.arange(nA+nB)

    for i in range(n_permutations):

        rng.shuffle(idx)

        perm[i] = energy_distance(
            Z[idx[:nA]],
            Z[idx[nA:]]
        )

    p = (np.sum(perm >= observed)+1)/(n_permutations+1)

    return {
        "energy_distance": observed,
        "p_value": p,
        "U": U,
        "V": V,
        "null_distribution": perm,
    }

In [7]:
result = eof2d_energy_test(
    geoclaw_data,
    hysea_data,
    n_space=25,
    n_time=30,
    n_permutations=500,
    random_state=42,
)

print(result["energy_distance"])
print(result["p_value"])

3.5227226933967657
0.001996007984031936
